# Przewodnik: co robimy w Colabie i jak analizujemy wyniki

Ten notebook jest mapą całego aktualnego przepływu pracy: od wrzucenia danych do Google Drive, przez trening i generowanie w Colabie, aż po analizę wyników i decyzję o następnym eksperymencie.

Nie jest to kolejny ciężki trening. To notebook kontrolny i wyjaśniający: pokazuje, **co robią notebooki Colab**, **jakie pliki powstają**, **jak czytać metryki** i **dlaczego po pełnym UnCLIP przeszliśmy do retrieval/reranking**.

## Najkrótszy stan projektu

Cel projektu: dążyć do rekonstrukcji obrazu z EEG. Aktualnie nie próbujemy już tylko klasyfikować kategorii, ale mapować EEG na reprezentacje obrazu i na tej podstawie odtwarzać albo wybierać obraz.

Co już wiemy z aktualnych wyników:

- EEGNet klasyfikacyjny daje sygnał ponad losowość: test accuracy około `21.3%` przy `9.1%` losowo.
- EEG → CLIP retrieval też jest ponad losowy: top-5 około `19.2%` przy `11.4%` losowo.
- Pełny Stable UnCLIP z embeddingu EEG działa technicznie, ale liczbowo przegrywa z VAE baseline.
- Stable UnCLIP oracle też nie bije VAE, więc ograniczeniem jest nie tylko EEG, ale też sam generator dla naszych bodźców.
- Następny sensowny kierunek to sprawdzić, czy EEG → embedding lepiej działa jako retrieval/reranking kandydatów niż jako bezpośrednie warunkowanie dyfuzji.

## Główne notebooki i ich rola

| Notebook | Rola | Kiedy używać |
|---|---|---|
| `TRENING_EEGNET_COLAB.ipynb` | sanity check klasyfikacyjny EEGNet | gdy chcemy sprawdzić, czy w danych EEG jest sygnał ponad losowość |
| `REKONSTRUKCJA_UNCLIP_COLAB.ipynb` | EEG → CLIP embedding → Stable UnCLIP | gdy chcemy wygenerować obrazy z EEG albo oracle CLIP embeddingów |
| `ANALIZA_WYNIKOW_COLAB_REKONSTRUKCJA.ipynb` | analiza wyników Colab | po pobraniu/utworzeniu wyników, żeby porównać smoke/full/VAE |
| `RETRIEVAL_RERANKING_PO_UNCLIP_COLAB.ipynb` | kolejny eksperyment po UnCLIP | gdy pełny UnCLIP nie bije VAE i chcemy sprawdzić retrieval/reranking |
| ten notebook | przewodnik i metaanaliza | gdy chcesz zrozumieć całość bez grzebania po wielu plikach |


## Co musi być na Google Drive

W Colabie zakładamy prywatny folder:

```text
MyDrive/biai/data/
MyDrive/biai/results/
```

W `MyDrive/biai/data/` powinny leżeć:

```text
biai_eeg_qc_0_0p8.zip
biai_unclip_assets.zip
```

`biai_eeg_qc_0_0p8.zip` to duża paczka z epokami EEG po QC. `biai_unclip_assets.zip` to mała paczka z obrazami, manifestami i skryptami potrzebnymi do rekonstrukcji/analizy.

Po lokalnej zmianie skryptów paczkę assets regenerujemy przez:

```powershell
.\scripts\prepare_unclip_colab_assets.ps1
```

Potem wrzucamy świeże `colab_export/biai_unclip_assets.zip` na Drive, nadpisując starą wersję.

## Przepływ Colab krok po kroku

```text
Google Drive ZIP-y
  ├─ biai_eeg_qc_0_0p8.zip        -> epoki EEG po QC
  └─ biai_unclip_assets.zip       -> obrazy, manifesty, skrypty
        ↓
Colab runtime /content/biai_unclip
        ↓
extract_unclip_image_embeddings.py
        ↓
image_embeddings_unclip_...       -> CLIP image embeddings + raw embeddings
        ↓
train_eeg_image_retrieval.py
        ↓
MyDrive/biai/results/unclip_mole_retrieval
        ↓
generate_unclip_from_eeg.py
        ↓
MyDrive/biai/results/unclip_mole_generation_smoke
MyDrive/biai/results/unclip_mole_generation_full
        ↓
notebooki analizy i kolejnego eksperymentu
```

Ważne rozróżnienie: `smoke` to tylko szybki test 2 obrazów. Pełna ocena zaczyna się dopiero od `unclip_mole_generation_full`.

## Co dokładnie analizuję

Analiza ma kilka warstw:

1. **Czy pipeline działa technicznie?** Patrzę, czy powstały summary JSON, CSV metryk, foldery `generated/`, `grids/`, checkpointy i logi.
2. **Czy EEG ma sygnał?** Patrzę na klasyfikację EEGNet oraz retrieval top-1/top-5/top-10 względem szansy losowej.
3. **Czy generator dobrze wykorzystuje informację?** Porównuję `eeg` vs `oracle`: jeżeli oracle jest słaby, problem leży też w generatorze.
4. **Czy wynik bije baseline?** Porównuję z lokalnym VAE ensemble po L1/PSNR/SSIM.
5. **Czy metryki zgadzają się z obrazami?** Oglądam gridy, bo SSIM potrafi premiować tło/kolor, nawet gdy kategoria jest nietrafiona.
6. **Jaki kolejny eksperyment ma sens?** Jeżeli UnCLIP nie bije VAE, nie dokładamy losowo większego generatora; sprawdzamy retrieval/reranking albo poprawę EEG→embedding.

In [ ]:
# Komórka startowa: importy i lokalizacja wyników.
from pathlib import Path
import json
import sys

import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

try:
    from IPython.display import Image, Markdown, display
except ModuleNotFoundError:
    class Markdown(str):
        pass

    class Image:
        def __init__(self, filename=None, **kwargs):
            self.filename = filename

        def __repr__(self):
            return f'<Image {self.filename}>'

    def display(value):
        print(value)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

PROJECT_ROOT = Path.cwd()
DRIVE_ROOT = Path('/content/drive/MyDrive')
PARTICIPANT = 'mole'

RESULT_ROOT_CANDIDATES = [
    PROJECT_ROOT / 'wyniki colab',
    PROJECT_ROOT / 'colab_results_20260626T204736Z_3_001',
    PROJECT_ROOT / 'colab_results',
    DRIVE_ROOT / 'biai' / 'results',
    DRIVE_ROOT / 'wyniki colab',
    DRIVE_ROOT / 'biai',
]

VAE_BASELINE = {
    'model': 'VAE ensemble (mole)',
    'images': 44,
    'l1': 0.2355921593579379,
    'psnr': 11.59615940397436,
    'ssim': 0.2864757523956624,
    'source': 'vae_participant_sweep_no_abc_20260626',
}

print('PROJECT_ROOT:', PROJECT_ROOT)
print('Sprawdzane miejsca z wynikami:')
for candidate in RESULT_ROOT_CANDIDATES:
    print(' -', candidate, 'OK' if candidate.exists() else 'brak')

In [ ]:
# Opcjonalnie w Colabie: jeśli wyniki są na Drive, a Drive nie jest zamontowany, spróbuj zamontować.
if Path('/content').exists() and not DRIVE_ROOT.exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)
        print('Drive zamontowany.')
    except Exception as exc:
        print('Nie udało się zamontować Drive:', exc)
        print('Jeśli pojawia się credential propagation unsuccessful: zrestartuj runtime, odłącz i usuń runtime, użyj jednego konta Google albo trybu incognito.')

In [ ]:
# Wyszukiwanie katalogów wyników.
def read_json(path):
    path = Path(path)
    if path.is_file():
        return json.loads(path.read_text(encoding='utf-8'))
    return None


def unique_by_name(paths):
    result = []
    seen = set()
    for path in sorted(paths):
        key = path.name
        if key not in seen:
            seen.add(key)
            result.append(path)
    return result


roots = [path for path in RESULT_ROOT_CANDIDATES if path.exists()]
generation_dirs = []
retrieval_dirs = []
eegnet_dirs = []

for root in roots:
    generation_dirs.extend(path for path in root.rglob(f'unclip_{PARTICIPANT}_generation_*') if path.is_dir())
    retrieval_dirs.extend(path for path in root.rglob(f'unclip_{PARTICIPANT}_retrieval') if path.is_dir())
    eegnet_dirs.extend(path for path in root.rglob(f'eegnet_{PARTICIPANT}_colab') if path.is_dir())

generation_dirs = unique_by_name(generation_dirs)
retrieval_dirs = unique_by_name(retrieval_dirs)
eegnet_dirs = unique_by_name(eegnet_dirs)

print('Generacja:')
for path in generation_dirs:
    print(' -', path)
print('\nRetrieval EEG→CLIP:')
for path in retrieval_dirs:
    print(' -', path)
print('\nEEGNet klasyfikacja:')
for path in eegnet_dirs:
    print(' -', path)

if not generation_dirs and not retrieval_dirs and not eegnet_dirs:
    raise FileNotFoundError('Nie znalazłem wyników. Umieść je w folderze wyniki colab albo na Drive w MyDrive/biai/results.')

In [ ]:
# Podsumowanie EEGNet i retrieval: czy EEG ma sygnał ponad losowość?
summary_rows = []

for eegnet_dir in eegnet_dirs:
    summary = read_json(eegnet_dir / 'eegnet_summary.json')
    if summary:
        summary_rows.append({
            'stage': 'EEGNet classification',
            'folder': eegnet_dir.name,
            'metric': 'test_accuracy',
            'value': summary.get('final_test_accuracy'),
            'chance_or_baseline': 1 / len(summary.get('labels', [])) if summary.get('labels') else None,
            'comment': f"best_epoch={summary.get('best_epoch')}",
        })

for retrieval_dir in retrieval_dirs:
    summary = read_json(retrieval_dir / 'retrieval_summary.json')
    if summary:
        test = summary.get('test', {})
        for metric in ['top1', 'top5', 'top10', 'category_top1']:
            chance_key = f'chance_{metric}'
            summary_rows.append({
                'stage': 'EEG→CLIP retrieval',
                'folder': retrieval_dir.name,
                'metric': metric,
                'value': test.get(metric),
                'chance_or_baseline': test.get(chance_key),
                'comment': f"best_epoch={summary.get('best_epoch')}, candidates={test.get('candidates')}",
            })

signal_df = pd.DataFrame(summary_rows)
display(signal_df)

In [ ]:
# Podsumowanie generacji UnCLIP: smoke vs full, EEG vs oracle.
generation_rows = []

for gen_dir in generation_dirs:
    summary = read_json(gen_dir / 'unclip_generation_summary.json')
    if not summary:
        continue
    run_type = 'full' if gen_dir.name.endswith('_full') else 'smoke' if gen_dir.name.endswith('_smoke') else gen_dir.name
    for variant in ['eeg', 'oracle']:
        metrics = summary.get(variant)
        if not metrics:
            continue
        generation_rows.append({
            'run': run_type,
            'variant': variant,
            'images': summary.get('images'),
            'steps': summary.get('num_inference_steps'),
            'l1': metrics.get('l1'),
            'psnr': metrics.get('psnr'),
            'ssim': metrics.get('ssim'),
            'folder': str(gen_dir),
        })

generation_df = pd.DataFrame(generation_rows)
display(generation_df.sort_values(['run', 'variant']).reset_index(drop=True))

In [ ]:
# Porównanie z VAE baseline. To jest główna tabela decyzyjna.
comparison_rows = [VAE_BASELINE]

if 'generation_df' in globals() and not generation_df.empty:
    for _, row in generation_df.iterrows():
        if row['run'] == 'full':
            comparison_rows.append({
                'model': f"Stable UnCLIP {row['variant']} full",
                'images': row['images'],
                'l1': row['l1'],
                'psnr': row['psnr'],
                'ssim': row['ssim'],
                'source': row['folder'],
            })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.sort_values('ssim', ascending=False).reset_index(drop=True))

In [ ]:
# Analiza per kategoria dla pełnego UnCLIP.
full_dirs = [path for path in generation_dirs if path.name.endswith('_full')]
if full_dirs:
    full_dir = full_dirs[0]
    metrics_csv = full_dir / 'unclip_metrics_per_image.csv'
    metrics_df = pd.read_csv(metrics_csv)
    per_category = metrics_df.groupby('image_category').agg(
        images=('image_id', 'count'),
        repetitions=('repetitions', 'sum'),
        eeg_ssim=('eeg_ssim', 'mean'),
        oracle_ssim=('oracle_ssim', 'mean'),
        eeg_l1=('eeg_l1', 'mean'),
        oracle_l1=('oracle_l1', 'mean'),
    ).sort_values('eeg_ssim', ascending=False)
    display(per_category)
else:
    print('Brak pełnego katalogu unclip_*_generation_full.')

In [ ]:
# Najlepsze i najgorsze przypadki pełnego UnCLIP EEG.
if 'metrics_df' in globals():
    display(Markdown('### Najlepsze przypadki EEG→UnCLIP według SSIM'))
    display(metrics_df.sort_values('eeg_ssim', ascending=False).head(10)[[
        'image_id', 'image_category', 'repetitions', 'eeg_ssim', 'oracle_ssim', 'eeg_l1', 'oracle_l1'
    ]])
    display(Markdown('### Najgorsze przypadki EEG→UnCLIP według SSIM'))
    display(metrics_df.sort_values('eeg_ssim').head(10)[[
        'image_id', 'image_category', 'repetitions', 'eeg_ssim', 'oracle_ssim', 'eeg_l1', 'oracle_l1'
    ]])
else:
    print('Najpierw uruchom komórkę per-kategoria.')

In [ ]:
# Podgląd gridów — tutaj często widać to, czego nie mówi sama liczba SSIM.
for gen_dir in generation_dirs:
    grid_dir = gen_dir / 'grids'
    grids = [grid_dir / 'best_eeg_unclip.jpg', grid_dir / 'best_oracle_unclip.jpg']
    existing = [grid for grid in grids if grid.is_file()]
    if not existing:
        continue
    display(Markdown(f'## {gen_dir.name}'))
    for grid in existing:
        display(Markdown(f'### {grid.name}'))
        display(Image(filename=str(grid)))

## Jak interpretuję metryki

- `L1`: średnia bezwzględna różnica pikseli. Mniej znaczy lepiej.
- `MSE`: średni błąd kwadratowy pikseli. Mniej znaczy lepiej.
- `PSNR`: metryka jakości rekonstrukcji pochodząca z MSE. Więcej znaczy lepiej.
- `SSIM`: podobieństwo strukturalne. Więcej znaczy lepiej, ale trzeba patrzeć na gridy, bo SSIM może być mylący przy obrazach semantycznych.
- `top1/top5/top10`: czy prawdziwy obraz jest wśród najbliższych kandydatów w przestrzeni embeddingów.
- `category_top1`: czy najbliższy kandydat ma tę samą kategorię co cel.

W tym projekcie sama metryka pikselowa nie wystarcza. Dla rekonstrukcji obrazu z EEG ważne są jednocześnie: kategoria, ogólny układ, podobieństwo wizualne i stabilność względem uczestników.

In [ ]:
# Automatyczny werdykt z obecnych wyników.
messages = []

if 'comparison_df' in globals() and not comparison_df.empty:
    best = comparison_df.sort_values('ssim', ascending=False).iloc[0]
    messages.append(f"Najlepszy wynik po SSIM: **{best['model']}** (`SSIM={best['ssim']:.3f}`).")
    unclip_eeg = comparison_df[comparison_df['model'].str.contains('UnCLIP eeg', case=False, na=False)]
    unclip_oracle = comparison_df[comparison_df['model'].str.contains('UnCLIP oracle', case=False, na=False)]
    if not unclip_eeg.empty:
        messages.append(f"Pełny UnCLIP EEG: `SSIM={float(unclip_eeg.iloc[0]['ssim']):.3f}`.")
    if not unclip_oracle.empty:
        messages.append(f"Pełny UnCLIP oracle: `SSIM={float(unclip_oracle.iloc[0]['ssim']):.3f}`.")
    if best['model'].startswith('VAE'):
        messages.append('Wniosek: obecny Stable UnCLIP nie bije VAE. Nawet oracle jest słabszy od VAE, więc ograniczeniem jest też generator, nie tylko EEG.')
        messages.append('Decyzja: kolejny notebook powinien testować retrieval/reranking albo poprawę dekodera EEG→embedding, zamiast po prostu odpalać więcej dyfuzji.')

if 'signal_df' in globals() and not signal_df.empty:
    retrieval_top5 = signal_df[(signal_df['stage'] == 'EEG→CLIP retrieval') & (signal_df['metric'] == 'top5')]
    if not retrieval_top5.empty:
        row = retrieval_top5.iloc[0]
        messages.append(f"Retrieval top-5: `{row['value']:.2%}` vs losowo `{row['chance_or_baseline']:.2%}` — sygnał jest, ale jest słaby/rozmyty.")

display(Markdown('\n\n'.join(messages) if messages else 'Brak danych do werdyktu.'))

## Dlaczego następnym krokiem jest retrieval/reranking

Pełny UnCLIP z EEG poprawił się względem smoke testu, ale nadal nie pobił VAE. Co ważniejsze: `oracle` też nie pobił VAE. To oznacza, że samo podanie lepszego embeddingu do Stable UnCLIP nie gwarantuje wiernej rekonstrukcji naszych bodźców.

W gridach widać dodatkowo, że EEG→UnCLIP często generuje obraz z niewłaściwej kategorii. Dlatego kolejny notebook (`RETRIEVAL_RERANKING_PO_UNCLIP_COLAB.ipynb`) sprawdza inną hipotezę:

> Może embedding EEG nie jest jeszcze wystarczająco dobry do generowania obrazu, ale jest wystarczająco dobry do zawężenia lub wyboru kandydatów.

Jeśli retrieval/reranking okaże się mocniejszy, następny etap to candidate-constrained generation: najpierw wybieramy kandydatów z EEG, potem dopiero generujemy albo składamy rekonstrukcję z silniejszym warunkowaniem.

## Typowe błędy Colaba i co oznaczają

### `FileNotFoundError: Brak pliku na Drive`

Colab nie widzi ZIP-a. Sprawdź ścieżki:

```text
/content/drive/MyDrive/biai/data/biai_eeg_qc_0_0p8.zip
/content/drive/MyDrive/biai/data/biai_unclip_assets.zip
```

Jeśli plik widzisz w przeglądarce Drive, a Colab nie: zrób `drive.mount('/content/drive', force_remount=True)` i upewnij się, że Colab jest zalogowany na to samo konto.

### `credential propagation was unsuccessful`

To błąd autoryzacji Google/Colab. Zwykle pomaga: `Runtime → Disconnect and delete runtime`, odświeżenie strony, tryb incognito, jedno konto Google, odblokowane cookies dla `colab.research.google.com`, `accounts.google.com`, `drive.google.com`.

### Colab otwiera zły branch GitHuba

Branch z ukośnikiem (`codex/eeg-pipeline-qc`) bywa źle parsowany przez Colab. Używamy aliasu bez ukośnika:

```powershell
git push origin HEAD:codex-eeg-pipeline-qc
```

i linków typu:

```text
https://colab.research.google.com/github/taf4you2/biai/blob/codex-eeg-pipeline-qc/notebooks/...
```

## Minimalna procedura po każdym nowym wyniku

1. Pobierz albo zostaw wyniki w `MyDrive/biai/results`.
2. Otwórz `ANALIZA_WYNIKOW_COLAB_REKONSTRUKCJA.ipynb`.
3. Sprawdź, czy wynik to `smoke`, czy `full`.
4. Porównaj z VAE baseline.
5. Obejrzyj gridy.
6. Jeśli generator przegrywa nawet w oracle — nie eskaluj bezmyślnie dyfuzji; sprawdź retrieval/reranking albo popraw EEG→embedding.
7. Jeśli retrieval/reranking wygra — projekt idzie w stronę candidate-constrained reconstruction.

W skrócie: nie chodzi o to, żeby produkować coraz więcej obrazków. Chodzi o to, żeby każdy wynik zawężał hipotezę, gdzie naprawdę jest wąskie gardło.